# Models&Tokenizers

[こちら](https://huggingface.co/learn/llm-course/en/chapter2/3)と[こちら](https://huggingface.co/learn/llm-course/en/chapter2/4)と[こちら](https://huggingface.co/learn/llm-course/en/chapter2/5)と[こちら](https://huggingface.co/learn/llm-course/en/chapter2/6)のチュートリアルを日本語で解説

前のNotobookで解説したpipelineはTokenizer、Model、Post Processing処理を内包しています。pipelineを使用せずに、これらの処理を個別実行する方法を解説します。

### Tokenizer

Tokenizerは入力テキストを単語/サブワードに分割し、整数値であるトークンIDを割り当てるの役割を果たします。
`transformers.AutoTokenizer.from_pretrained`メソッドを用いることで、指定したチェックポイント（トークナイザ名に相当）に応じて適切なTokenizerのインスタンスを取得できます。

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

あるいは、明示的にトークナイザの種類に対応したクラスを使用するのもOKです（こちらの方が可読性や思わぬ動作を防ぐという観点では良い方法です）

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained(checkpoint)

入力テキストまたはそのリストをtokenizerに渡すことで、トークンIDに変換することができます。`return_tensors`引数に"pt"を指定しているため、torch.Tensor形式の以下dictが返ってきます（"np"を指定するとnumpy.ndarray形式になる）。

なお、各torch.Tensorは二重リスト構造になっていることにご注意ください、これはTransformersのModelは基本的にバッチ単位での入力を想定していることに由来します（詳細は後述）。

In [ ]:
sequence = "Using a Transformer network is simple"
inputs = tokenizer(sequence, return_tensors="pt")
print(inputs)

`tokenizer.tokenize`メソッドを使用することで、対応する単語でtokenize結果を取得することもできます（`tokenize`メソッドでは[CLS]や[SEP]のような特殊トークンが付加されないことにご注意ください）。

In [ ]:
inputs = tokenizer.tokenize(sequence, return_tensors="pt")
print(inputs)


以下のケースでは2つの入力テキストをリストで渡しており、以下内容が出力されます。
- 'input_ids': 変換後のトークンIDのリスト（サイズ：リスト中のテキストの最大トークン数N × テキスト数）
- 'token_type_ids': 各トークンが「何番目の文章（セグメント）に属しているか」を示すインデックス。エンコーダ型モデルで「2つの文章の関係性」を扱うタスクで使用するが、今回は不使用（サイズ：リスト中のテキストの最大トークン数N × テキスト数）。後述
- 'attention_mask': トークンIDのうち有効な部分（サイズ：リスト中のテキストの最大トークン数N × テキスト数）

リスト中のテキストの最大トークン数Nはシーケンス長、テキスト数はバッチサイズとも呼ばれます。

In [ ]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

`decode`メソッドでトークンIDを文字に変換することもできます（CLSとSEPトークンが追加されていることが見て取れます）

In [ ]:
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
decoded_string = tokenizer.decode(inputs["input_ids"][0].tolist())
print(decoded_string)

複数のテキストをリスト渡しではなく別々の引数として渡すと、一体のシーケンスとして読み込まれた上で異なる'token_type_ids'が割り振られます。また2つの分の間に[SEP]トークンが挿入されます。2つの入力を比較するようなタスクでは、この[SEP]トークンの挿入は重要な役割を果たします。

In [ ]:
inputs = tokenizer(raw_inputs[0], raw_inputs[1], return_tensors="pt")
print(inputs)
print(tokenizer.decode(inputs["input_ids"][0].tolist()))

#### paddingとtruncate

デフォルトの`padding=True`では、バッチ内の最も長い`padding="longest"`に相当するバッチ内の最も長いシーケンスに合わせたパディングが実施されますが、`padding="max_length"`を指定するとトークナイザの最大長、または`max_length`引数で指定された長さまでパディングされるようになります

In [ ]:
# Will pad the sequences up to the maximum sequence length
model_inputs = tokenizer(raw_inputs, padding="longest")
print(model_inputs)

# Will pad the sequences up to the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(raw_inputs, padding="max_length")
print(model_inputs)

# Will pad the sequences up to the specified max length
model_inputs = tokenizer(raw_inputs, padding="max_length", max_length=8)
print(model_inputs)

`padding=False`を指定すると異なる長さのテキストで異なる長さの出力が帰るようになります（ただし、return_tensorsに"pt"を指定するとエラーが出る上、TransfromersのLLMモデルは基本的にバッチ内のシーケンス長が等しい長方形形状のTensor入力を想定しているため、LLMの前処理として使うケースは少ないでしょう）

In [ ]:
inputs = tokenizer(raw_inputs, padding=False, truncation=True)
print(inputs)

`truncation=True`を指定すると、シーケンス長上限（トークナイザの最大長または`max_length`引数で指定）を超えた分のトークンが切り捨てられます。このとき、[SEP]や[CLS]等の重要な特殊トークンは基本的に削除されず、末尾等の適切な位置にくるよう再調整されます

In [ ]:
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

# Will truncate the sequences that are longer than the model max length
# (512 for BERT or DistilBERT)
model_inputs = tokenizer(sequences, truncation=True)
print(model_inputs)

# Will truncate the sequences that are longer than the specified max length
model_inputs = tokenizer(sequences, max_length=8, truncation=True)
print(model_inputs)

#### Tokenizerの保存

`save_pretrained`メソッドでトークナイザを保存できます。

In [ ]:
tokenizer.save_pretrained(f"/workspace/models/tokenizer/{checkpoint}" )

### Model

LLMモデルは`transformers.AutoModel.from_pretrained`メソッドを用いることで、指定したチェックポイント（HuggingFaceのモデルタグ名）に応じて適切なモデルのインスタンスを取得できます。

In [ ]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

あるいは、明示的にモデルの種類に対応したクラスを使用するのもOKです（こちらの方が可読性や思わぬ動作を防ぐという観点では良い方法です）

In [ ]:
from transformers import DistilBertModel

model = DistilBertModel.from_pretrained(checkpoint)

modelインスタンスに先ほどのTokenizerの出力を渡すことで、モデルの推論を実行できます（以下の例では"input_ids"、"token_type_ids"、"attention_mask"の3引数を渡していますが、基本的には"input_ids"と"attention_mask"を渡すだけで十分で、また"attention_mask"が全て1である前提であれば"input_ids"のみを渡すだけで推論できます）。

出力の形式はモデルの種類ごとに異なりますが、特に以下のメンバ変数に格納された出力が重要となります

エンコーダ型モデル（BERT等）：`last_hidden_state`に格納された最終層の特徴ベクトル（形状：バッチサイズ × シーケンス長N × 特徴ベクトル長D）
デコーダ型生成モデル（GPT等）や分類モデル：`logits`に格納されているロジット値（形状：バッチサイズ × クラス数、テキスト生成ではクラス数=トークンのカテゴリ数となる）

In [ ]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)
print(outputs.last_hidden_state)

In [ ]:
from transformers import AutoModelForSequenceClassification 

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english" 
model = AutoModelForSequenceClassification.from_pretrained(checkpoint) 
outputs = model(**inputs)
print(outputs.logits.shape)
print (outputs.logits)

`save_pretrained`メソッドでモデルを保存できます（設定ファイルであるconfig.jsonと、モデルの重みである.safetensorファイルに分けて保存されます）。

In [ ]:
model.save_pretrained(f"/workspace/models/llms/decoders/{checkpoint}")

保存したパスを`from_pretrained`に指定することでモデルを読込できます。

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(f"/workspace/models/llms/decoders/{checkpoint}") 
outputs = model(**inputs)
print(outputs.logits.shape)
print (outputs.logits)

なお、HuggingFace Hubからモデルを`from_pretrained`で読み込んだり、`push_to_hub`メソッドでHubにアップロードすることもできます

```python
from transformers import AutoModel

model = AutoModel.from_pretrained("your-username/my-awesome-model")
model.push_to_hub("my-awesome-model")
```

#### Modelに渡すべき形式

TransformersのModelは基本的にバッチ単位での入力を想定しています。よって以下のように単リスト（Sequence次元のみ）のtorch.TensorをModelに渡すとエラーが出ます。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokens = tokenizer.tokenize(sequence)
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)
print(input_ids)
# This line will fail.
model(input_ids)

以下のように2次元リストのtorch.Tensor（BatchとSequenceの次元）を渡せば推論できます

In [ ]:
input_ids = torch.tensor([ids])
print(input_ids)
# This line will fail.
output = model(input_ids)
print("Logits:", output.logits)

### Post Processing処理

モデルの出力はそのままでは活用しづらい形式となっているため、後処理を行なって目当ての値に変換します。

例えばデコーダ型生成モデル（GPT等）や分類モデルでは、モデルが出力するロジット値をクラス確率やクラス判定（テキスト出力）に変換することが一般的です。
以下の例では、ロジット値にソフトマックス関数を適用することで、クラス確率に変換しています。

In [ ]:
import torch 

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

クラス確率をクラス判定に変換するには、`argmax`メソッドでクラス確率が最大のクラスを特定した上で、`model.config.id2label`で取得したクラス名を適用します。

In [ ]:
# クラス確率をクラス判定に変換
predicted_class_ids = predictions.argmax(dim=-1).tolist()
predicted_labels = [model.config.id2label[class_id] for class_id in predicted_class_ids]

print("predicted_class_ids:", predicted_class_ids)
print("predicted_labels:", predicted_labels)